# Confidence intervals: the z-procedure
A 95% confidence interval for a mean when the population standard deviation is known, critical values from the standard normal curve, and the confidence intervals seaborn draws as error bars.

In [ ]:
import numpy as np
import pandas as pd
import seaborn.objects as so
from scipy import stats

## 1. The z-procedure by hand: subscribers' mean age
Sample of n = 100, sample mean 28 years, population standard deviation known to be 15 years.

In [ ]:
x_bar, sigma, n = 28, 15, 100
level = 0.95
alpha = 1 - level
z = stats.norm.ppf(1 - alpha / 2)          # area 0.975 to the left
se = sigma / np.sqrt(n)                    # standard error
margin = z * se                            # margin of error
print(f"z = {z:.4f}, SE = {se}, margin of error = {margin:.2f}")
print(f"95% CI: {x_bar - margin:.2f} to {x_bar + margin:.2f}")
# scipy does the same in one call
print(stats.norm.interval(level, loc=x_bar, scale=se))

## 2. Critical values for common confidence levels

In [ ]:
for level in [0.5, 0.75, 0.8, 0.9, 0.95, 0.99]:
    print(f"{level:.0%}: z = {stats.norm.ppf((1 + level) / 2):.3f}")

## 3. Checking the derivation by simulation
If the population is N(28, 15^2), the share of samples whose interval contains 28 should be 95%.

In [ ]:
rng = np.random.default_rng(42)            # fixed seed
means = rng.normal(28, 15, size=(100_000, 100)).mean(axis=1)
z_scores = (means - 28) / (15 / np.sqrt(100))
print("share with -1.96 < Z < 1.96:", np.mean(np.abs(z_scores) < 1.96))
print("share of intervals containing 28:", np.mean(np.abs(means - 28) <= 1.96 * 1.5))

## 4. Seaborn's error bars are confidence intervals

In [ ]:
df = pd.read_csv("data/titanic_train.csv")
# bars = mean fare per class; black ranges = 95% confidence intervals (bootstrap)
(so.Plot(df, x="Pclass", y="Fare")
   .add(so.Bar(), so.Agg("mean"))
   .add(so.Range(color="black"), so.Est("mean", errorbar=("ci", 95), seed=0))
   .show())
# the same intervals with the normal approximation x_bar ± 1.96 s / sqrt(n)
for cls, g in df.groupby("Pclass")["Fare"]:
    se = g.std() / np.sqrt(len(g))
    print(f"class {cls}: n = {len(g)}, mean {g.mean():.2f}, 95% CI {g.mean() - 1.96 * se:.2f} to {g.mean() + 1.96 * se:.2f}")